# Test AgentCore Deployment

This notebook tests the deployed Context-Aware Agent on AgentCore Runtime.

**Prerequisites:**
1. AgentCore infrastructure deployed (`AgentCoreContext` stack)
2. Agent deployed via `deploy_runtime.py`
3. Neo4j seeded with sample data

## 1. Setup and Verification

In [ ]:
import boto3
import json
from datetime import datetime

# AWS configuration
region = "us-east-1"
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=region)
cfn = boto3.client("cloudformation", region_name=region)

print(f"✅ AWS SDK configured for region: {region}")

In [ ]:
# Get AgentCore Runtime ARN from CloudFormation (if exported)
# AgentCore runtimes are NOT CloudFormation resources, they're API-managed

# List available AgentCore runtimes
try:
    response = bedrock_agent_runtime.list_agent_runtimes()
    print("📋 Available AgentCore Runtimes:")
    for runtime in response.get("agentRuntimes", []):
        print(f"  - {runtime['agentRuntimeArn']}")
        print(f"    Name: {runtime.get('agentName', 'N/A')}")
        print(f"    Status: {runtime.get('status', 'N/A')}")
        print()

## 2. Find Our Agent

AgentCore Runtime SDK deploys agents as managed runtimes. We need to find our `context_aware_agent`.

In [ ]:
# Find our agent by name
agent_runtime_arn = None
agent_name = "context_aware_agent"

try:
    response = bedrock_agent_runtime.list_agent_runtimes()
    for runtime in response.get("agentRuntimes", []):
        if runtime.get("agentName") == agent_name:
            agent_runtime_arn = runtime["agentRuntimeArn"]
            print(f"✅ Found agent: {agent_name}")
            print(f"   ARN: {agent_runtime_arn}")
            print(f"   Status: {runtime.get('status', 'N/A')}")
            break
    
    if not agent_runtime_arn:
        print(f"❌ Agent '{agent_name}' not found.")
        print("\nMake sure to run: python3 deploy_runtime.py")
except Exception as e:
    print(f"❌ Error listing runtimes: {e}")
    print("\nNote: bedrock-agent-runtime API may not have list_agent_runtimes() method.")
    print("If deployed via Runtime SDK, you should have the ARN from deploy output.")

## 3. Alternative: Use Runtime ARN from Deploy Output

If the above fails, paste the ARN from `deploy_runtime.py` output here:

In [ ]:
# Paste ARN from deploy_runtime.py output
# agent_runtime_arn = "arn:aws:bedrock:us-east-1:222634367169:agent-runtime/xxxxx"

if agent_runtime_arn:
    print(f"✅ Using agent runtime: {agent_runtime_arn}")
else:
    print("❌ Please set agent_runtime_arn above")

## 4. Invoke Agent - Test 1: Entity Extraction

Test if the agent can extract entities from a conversation.

In [ ]:
def invoke_agent(agent_arn: str, prompt: str, session_id: str = None):
    """Invoke AgentCore Runtime."""
    if not session_id:
        session_id = f"test-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    
    print(f"📤 Invoking agent...")
    print(f"   Session: {session_id}")
    print(f"   Prompt: {prompt}")
    print()
    
    try:
        response = bedrock_agent_runtime.invoke_agent(
            agentRuntimeArn=agent_arn,
            sessionId=session_id,
            inputText=prompt
        )
        
        # AgentCore returns streaming response
        completion = ""
        for event in response.get("completion", []):
            if "chunk" in event:
                chunk = event["chunk"]
                if "bytes" in chunk:
                    text = chunk["bytes"].decode("utf-8")
                    completion += text
                    print(text, end="", flush=True)
        
        print("\n")
        return completion, session_id
    
    except Exception as e:
        print(f"❌ Error invoking agent: {e}")
        return None, session_id

In [ ]:
# Test 1: Entity extraction
if agent_runtime_arn:
    response, session_id = invoke_agent(
        agent_runtime_arn,
        "I just met Sarah Chen from Acme Corporation. She's the VP of Engineering and we discussed their new AI platform project."
    )
else:
    print("⚠️  Set agent_runtime_arn first")

## 5. Test 2: Query Knowledge Graph

Test if the agent can recall information from Neo4j.

In [ ]:
# Test 2: Query existing context
if agent_runtime_arn and session_id:
    response, _ = invoke_agent(
        agent_runtime_arn,
        "What do you know about Sarah Chen?",
        session_id=session_id  # Continue same session
    )
else:
    print("⚠️  Run Test 1 first")

## 6. Test 3: Multi-Turn Conversation

Test memory persistence across multiple turns.

In [ ]:
# Test 3: Multi-turn conversation
if agent_runtime_arn:
    new_session = f"test-multiturn-{datetime.now().strftime('%H%M%S')}"
    
    # Turn 1: Introduce entity
    print("=" * 60)
    print("TURN 1: Introduce entity")
    print("=" * 60)
    invoke_agent(
        agent_runtime_arn,
        "I attended AWS Summit 2026 in San Francisco last week.",
        session_id=new_session
    )
    
    # Turn 2: Add related entity
    print("\n" + "=" * 60)
    print("TURN 2: Add related entity")
    print("=" * 60)
    invoke_agent(
        agent_runtime_arn,
        "I met John Smith there. He works at TechCorp.",
        session_id=new_session
    )
    
    # Turn 3: Query relationships
    print("\n" + "=" * 60)
    print("TURN 3: Query relationships")
    print("=" * 60)
    invoke_agent(
        agent_runtime_arn,
        "Who did I meet at AWS Summit?",
        session_id=new_session
    )
else:
    print("⚠️  Set agent_runtime_arn first")

## 7. Verify Neo4j Data

Check that entities were actually stored in Neo4j (via query_graph Lambda).

In [ ]:
# Get Lambda function ARN from stack
lambda_client = boto3.client("lambda", region_name=region)

try:
    response = cfn.describe_stacks(StackName="Neo4jContextGraph")
    outputs = response["Stacks"][0]["Outputs"]
    
    query_function_arn = None
    for output in outputs:
        if output["OutputKey"] == "QueryGraphFunctionArn":
            query_function_arn = output["OutputValue"]
            break
    
    if query_function_arn:
        print(f"✅ Found query Lambda: {query_function_arn}")
        
        # Query all Person entities
        response = lambda_client.invoke(
            FunctionName=query_function_arn,
            InvocationType="RequestResponse",
            Payload=json.dumps({
                "action": "query_entities",
                "entity_type": "Person",
                "actor_id": "demo-user-001"
            })
        )
        
        result = json.loads(response["Payload"].read())
        body = json.loads(result["body"])
        
        print(f"\n📊 Entities in Neo4j:")
        print(json.dumps(body["entities"], indent=2))
    else:
        print("❌ QueryGraphFunctionArn not found in stack outputs")
        
except Exception as e:
    print(f"❌ Error: {e}")

## 8. Verify DynamoDB Conversation History

Check that conversation history is stored in DynamoDB (Short-Term Memory).

In [ ]:
# Get DynamoDB table name from stack
dynamodb = boto3.resource("dynamodb", region_name=region)

try:
    response = cfn.describe_stacks(StackName="AgentCoreContext")
    outputs = response["Stacks"][0]["Outputs"]
    
    conversation_table_name = None
    for output in outputs:
        if output["OutputKey"] == "ConversationTableName":
            conversation_table_name = output["OutputValue"]
            break
    
    if conversation_table_name:
        print(f"✅ Found conversation table: {conversation_table_name}")
        
        table = dynamodb.Table(conversation_table_name)
        response = table.scan(Limit=10)
        
        print(f"\n💬 Recent conversations ({response['Count']} items):")
        for item in response["Items"]:
            print(f"  - Session: {item.get('session_id', 'N/A')}")
            print(f"    Role: {item.get('role', 'N/A')}")
            print(f"    Content: {item.get('content', 'N/A')[:100]}...")
            print()
    else:
        print("❌ ConversationTableName not found in stack outputs")
        
except Exception as e:
    print(f"❌ Error: {e}")

## 9. Summary & Diagnostics

Check overall system health.

In [ ]:
print("=" * 60)
print("SYSTEM HEALTH CHECK")
print("=" * 60)

# Check Neo4j stack
try:
    response = cfn.describe_stacks(StackName="Neo4jContextGraph")
    status = response["Stacks"][0]["StackStatus"]
    print(f"✅ Neo4j Stack: {status}")
except Exception as e:
    print(f"❌ Neo4j Stack: {e}")

# Check AgentCore stack
try:
    response = cfn.describe_stacks(StackName="AgentCoreContext")
    status = response["Stacks"][0]["StackStatus"]
    print(f"✅ AgentCore Stack: {status}")
except Exception as e:
    print(f"❌ AgentCore Stack: {e}")

# Check AgentCore Runtime
if agent_runtime_arn:
    print(f"✅ Agent Runtime: {agent_runtime_arn}")
else:
    print(f"❌ Agent Runtime: Not found or not deployed")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("")
print("If all checks pass:")
print("  1. ✅ Agent is working correctly")
print("  2. ✅ Ready for production use")
print("")
print("If any checks fail:")
print("  1. Check deploy_runtime.py output for errors")
print("  2. Check CloudWatch Logs: /aws/bedrock/agent-runtime/context_aware_agent")
print("  3. Verify Neo4j is running: aws ecs list-tasks --cluster Neo4j*")
print("  4. Check Lambda logs: /aws/lambda/Neo4jContextGraph-*")